# 03 · Metadata-only audit

**Project:** Acquisition shortcuts in public cancer imaging (authors: see README.md)
**Notebook written:** 28 Oct 2026 · **Needs:** `results/02_tasks_features/` · **Plan:** sections 7, 7a and **7b** (this analysis, fixed before it was run)

**What this notebook does, in plain words**

For each of the 26 tasks, it asks: **can the scan's "label card" alone (machine, software, settings, description words) guess the
patient's diagnosis class, without looking at the image?**
1. **Main answer (RQ1):** a simple model (logistic regression) learns from the label card in 5-fold cross-validation, repeated
   5 times. We report how well it guesses (AUC: 0.5 = coin toss, 1.0 = perfect) with a 95% confidence interval, and test it against
   chance by shuffling the labels 999 times. A task is **flagged** when the lower end of the interval is at least 0.60 and the
   corrected p-value (q) is below 0.05.
2. **Which clue (RQ2):** each part of the label card alone, each part left out, and each part scrambled.
3. **Hospital or settings (RQ3):** where the hospital is known, does the clue still work **between patients of the same hospital**?
4. **Checks:** a second model type (LightGBM), adding file size, the care-pathway clues (how much imaging, when), and the
   average AUC under shuffled labels (should be about 0.5: proof that the pipeline does not leak).

**No images are used.** Outputs go to `results/03_audit/`.

**How to run:** in Colab, *Runtime → Run all*. No GPU is needed. Sign in with **the Google account that holds the project Drive folder** when Drive asks.
Expect roughly half an hour to an hour; keep the browser tab open.

In [1]:
# 1. Setup: Drive and folders
import os, re, sys, json, time, hashlib, datetime, platform, warnings
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
assert PROJECT.exists(), (f'Project folder not found: {PROJECT}. '
                          'Is Colab signed in to the Drive of the Google account that holds the project Drive folder?')

IN02 = PROJECT / 'results' / '02_tasks_features'
OUT = PROJECT / 'results' / '03_audit'
assert (IN02 / 'run_manifest.json').exists(), 'Run notebook 02 first: results/02_tasks_features/ is missing.'
OUT.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
print('Project folder:', PROJECT)
print('Outputs to    :', OUT)
print('Run started   :', RUN_STARTED, '(UTC)')

Mounted at /content/drive
Project folder: /content/drive/MyDrive/Acquisition-Shortcut-Audit
Outputs to    : /content/drive/MyDrive/Acquisition-Shortcut-Audit/results/03_audit
Run started   : 2026-10-28T16:27:39+00:00 (UTC)


In [2]:
# 2. Packages
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import lightgbm
from lightgbm import LGBMClassifier
import joblib
from joblib import Parallel, delayed

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

print('numpy', np.__version__, '| pandas', pd.__version__, '| scikit-learn', sklearn.__version__,
      '| lightgbm', lightgbm.__version__, '| CPUs', os.cpu_count())

numpy 2.1.3 | pandas 2.2.3 | scikit-learn 1.6.1 | lightgbm 4.6.0 | CPUs 2


## Frozen settings (plan section 7b)
Change them only with a dated entry in `00_LOG.md`; any change after this run is a post hoc analysis.

In [3]:
# 3. Frozen settings
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5          # 5 x 5 repeated stratified CV
N_BOOT = 2000                       # bootstrap resamples for 95% CIs
N_PERM = 999                        # label permutations for the test against chance
MIN_COUNT = 3                       # a category or a word is used only if >= 3 training patients have it
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05  # rule 7a.2
N_SHUFFLE = 5                       # shuffles per fold for the group-shuffle importance
MIN_PER_CLASS_SITE_REFIT = 10       # 7b.10(d)

LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
LGBM_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=10, subsample=0.8,
                   subsample_freq=1, colsample_bytree=0.8, class_weight='balanced', random_state=SEED,
                   verbose=-1, n_jobs=1)

# Scanner/protocol feature groups (primary model, 7b.3)
SCANNER_GROUPS = {
    'scanner (vendor, model)': ['vendor', 'model'],
    'software version': ['software_version'],
    'image count': ['image_count'],
    'slice thickness and kernel': ['slice_thickness_mm', 'kernel'],
    'body part': ['body_part'],
    'description words': ['protocol_text'],
}
SCANNER_COLS = [c for cols in SCANNER_GROUPS.values() for c in cols]
CARE_NUMERIC = ['cp_n_studies', 'cp_n_series', 'cp_n_series_CT', 'cp_n_series_MR', 'cp_has_PT', 'cp_n_modalities',
                'cp_n_studies_this_modality', 'cp_study_rank_used', 'cp_days_from_diagnosis']
CARE_COLS = CARE_NUMERIC + ['care_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part', 'site'}
LOG_NUMERIC = {'image_count', 'file_size_mb'}
TEXT = {'protocol_text', 'care_text'}

# Local code tests only (ignored in Colab): shuffled labels, fewer tasks, fewer resamples
TEST = {}
if not IN_COLAB:
    TEST = {'shuffle_labels': os.environ.get('ASA_TEST_SHUFFLE_LABELS') == '1',
            'tasks': [t for t in os.environ.get('ASA_TEST_TASKS', '').split(';') if t]}
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [4]:
# 4. Load and verify the inputs from notebook 02
man02 = json.loads((IN02 / 'run_manifest.json').read_text())
bad = [f['file'] for f in man02['outputs'] if md5(IN02 / f['file']) != f['md5']]
assert not bad, f'These notebook 02 files changed since that run: {bad}'
print('Notebook 02 outputs verified:', len(man02['outputs']), 'files, run of', man02['run_started_utc'])

tasks = pd.read_csv(IN02 / 'tasks_final.csv')
labels = pd.read_csv(IN02 / 'labels_long.csv')
feat = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')

KEY = ['collection', 'PatientID', 'modality']
text = (toks.assign(group=np.where(toks.category == 'care pathway', 'care_text', 'protocol_text'))
            .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
feat['cp_has_PT'] = feat['cp_has_PT'].astype(float)
for c in CARE_NUMERIC:
    if c not in feat:
        feat[c] = np.nan
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution

family = tasks[tasks.eligible].task_id.tolist()
if TEST.get('tasks'):
    family = [t for t in family if t in TEST['tasks']]
print(f'{len(family)} tasks in the family (7b.1)')

def task_data(tid):
    lab = labels[labels.task_id == tid]
    df = lab.merge(feat, on=KEY, how='left', validate='one_to_one')
    assert df.image_count.notna().all(), f'features missing for some patients of {tid}'
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if TEST.get('shuffle_labels'):
        y = np.random.default_rng(999).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    return df.reset_index(drop=True), y, classes

Notebook 02 outputs verified: 11 files, run of 2026-10-28T15:29:58+00:00
26 tasks in the family (7b.1)


## Model building blocks
All encoding happens inside each training fold, so no information from the test patients is used.

In [5]:
# 5. Encoding, models and fast AUC
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def make_model(cols, kind='lr'):
    est = LogisticRegression(**LR_PARAMS) if kind == 'lr' else LGBMClassifier(**LGBM_PARAMS)
    return Pipeline([('prep', column_transformer(cols)), ('model', est)])

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    """Binary AUC, or macro one-vs-rest AUC for more than two classes. P: n x k probabilities."""
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    out = []
    for r in range(N_REPEATS):
        skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r)
        out.append(list(skf.split(np.zeros(len(y)), y)))
    return out

In [6]:
# 6. Cross-validation engine
def _quiet():
    # LightGBM 4.6 with scikit-learn 1.6 warns about feature names on every prediction; it does not affect results
    warnings.filterwarnings('ignore', message='X does not have valid feature names')

def _fit_fold(df, y, cols, kind, train, test, k, shuffle_groups):
    _quiet()
    m = make_model(cols, kind).fit(df.iloc[train], y[train])
    proba = np.zeros((len(test), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[test])
    shuffled = {}
    if shuffle_groups:
        rng = np.random.default_rng(SEED + int(test[0]))
        for g, gcols in shuffle_groups.items():
            reps = []
            for _ in range(N_SHUFFLE):
                Xs = df.iloc[test].copy()
                perm = rng.permutation(len(test))
                Xs[gcols] = Xs[gcols].iloc[perm].to_numpy()
                ps = np.zeros((len(test), k))
                ps[:, m.classes_] = m.predict_proba(Xs)
                reps.append(ps)
            shuffled[g] = reps
    return test, proba, shuffled

def cv_oof(df, y, cols, kind='lr', splits=None, shuffle_groups=None):
    """Out-of-fold probabilities, shape (repeats, n, k). Optionally also with one group shuffled in the test fold."""
    k = int(y.max()) + 1
    splits = splits or repeated_splits(y)
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold)(df, y, cols, kind, tr, te, k, shuffle_groups) for r, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    S = {g: np.zeros((N_SHUFFLE, len(splits), len(y), k)) for g in (shuffle_groups or {})}
    for (r, _, _), (te, proba, shuffled) in zip(jobs, res):
        P[r, te] = proba
        for g, reps in shuffled.items():
            for s, ps in enumerate(reps):
                S[g][s, r, te] = ps
    return P, S

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P, rng=None, strata=None):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples (within strata if given)."""
    rng = rng or np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    idx_all = np.arange(len(y))
    groups = [idx_all] if strata is None else [idx_all[strata == s] for s in np.unique(strata)]
    while len(vals) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def estimate(df, y, cols, kind='lr', splits=None):
    P, _ = cv_oof(df, y, cols, kind, splits)
    lo, hi = boot_ci(y, P)
    return {'auc': mean_auc(y, P), 'ci_low': lo, 'ci_high': hi}, P

In [7]:
# 7. Permutation test (7b.6). One fixed 5-fold split that does not depend on the labels, so the test is exact.
#    The encoding uses no labels, so it is fitted once per training fold; each permutation refits only the model.
from sklearn.model_selection import KFold

def _encoded_folds(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def _single_cv_auc(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return auc(y, P)

def _perm_chunk(folds, ys):
    return [_single_cv_auc(folds, yp) for yp in ys]

def permutation_test(df, y, cols):
    folds = _encoded_folds(df, cols)
    observed = _single_cv_auc(folds, y)
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    p = (1 + int((null >= observed).sum())) / (1 + N_PERM)
    return observed, p, null

In [8]:
# 8. Within-site AUC (7b.10c): share of positive-negative pairs from the same site that the score orders correctly
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:   # no site holds both classes
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

## The audit
One pass per task. Each finished task is saved at once in `per_task/`. **If Colab disconnects, just run all cells again:**
finished tasks are loaded, not recomputed (only if they were made with exactly the same settings).

In [9]:
# 9. Run the audit over the family
PER_TASK = OUT / 'per_task'
PER_TASK.mkdir(exist_ok=True)
SETTINGS_KEY = hashlib.md5(json.dumps(
    {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
     'MIN_COUNT': MIN_COUNT, 'N_SHUFFLE': N_SHUFFLE, 'LR': LR_PARAMS, 'LGBM': LGBM_PARAMS, 'TEST': TEST,
     'inputs': man02['run_started_utc']}, sort_keys=True, default=str).encode()).hexdigest()

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

primary_rows, group_rows, site_rows, refit_rows, oof_rows = [], [], [], [], []
t_all = time.time()
for i, tid in enumerate(family, 1):
    t0 = time.time()
    stem = re.sub(r'\W+', '_', tid).strip('_')
    saved = PER_TASK / f'{stem}.json'
    if saved.exists():
        rec = json.loads(saved.read_text())
        if rec.get('settings_key') == SETTINGS_KEY:
            primary_rows.append(rec['primary']); group_rows += rec['groups']
            site_rows += rec['sites']; refit_rows += rec['refit']
            oof_rows.append(pd.read_csv(PER_TASK / f'{stem}_oof.csv.gz'))
            print(f'{i:>2}/{len(family)} {tid:<44} loaded from an earlier run')
            continue
    n_group, n_site, n_refit = len(group_rows), len(site_rows), len(refit_rows)
    df, y, classes = task_data(tid)
    k = len(classes)
    row = {'task_id': tid, 'task_type': tasks.set_index('task_id').task_type[tid], 'n_patients': len(y),
           'classes': ' | '.join(f'{c}: {int((y == j).sum())}' for j, c in enumerate(classes)), 'n_classes': k}

    # Primary model, with the group-shuffle importance computed on the same fits (7b.5, 7b.9)
    splits = repeated_splits(y)
    P, S = cv_oof(df, y, SCANNER_COLS, 'lr', splits, shuffle_groups=SCANNER_GROUPS)
    base = mean_auc(y, P)
    lo, hi = boot_ci(y, P)
    row.update({'auc': base, 'ci_low': lo, 'ci_high': hi})
    oof = pd.concat([pd.DataFrame({'task_id': tid, 'repeat': r, 'PatientID': df.PatientID, 'label': y,
                                   **{f'p_{j}': P[r][:, j] for j in range(k)}}) for r in range(N_REPEATS)],
                    ignore_index=True)

    # Test against chance (7b.6) and the null mean (S4)
    obs, p, null = permutation_test(df, y, SCANNER_COLS)
    row.update({'perm_observed_auc_single_cv': obs, 'perm_p': p, 'null_mean_auc': float(null.mean()),
                'null_95th_auc': float(np.percentile(null, 95))})

    # Secondary analyses (7b.8)
    for name, cols, kind in (('lgbm', SCANNER_COLS, 'lgbm'), ('plus_filesize', SCANNER_COLS + ['file_size_mb'], 'lr'),
                             ('care_only', CARE_COLS, 'lr'), ('scanner_plus_care', SCANNER_COLS + CARE_COLS, 'lr')):
        est, _ = estimate(df, y, cols, kind, splits)
        row.update({f'{name}_auc': est['auc'], f'{name}_ci_low': est['ci_low'], f'{name}_ci_high': est['ci_high']})

    # Which fields (7b.9): alone, left out, shuffled
    for g, gcols in SCANNER_GROUPS.items():
        alone, _ = estimate(df, y, gcols, 'lr', splits)
        left, _ = cv_oof(df, y, [c for c in SCANNER_COLS if c not in gcols], 'lr', splits)
        shuf = np.mean([mean_auc(y, S[g][s]) for s in range(N_SHUFFLE)])
        group_rows.append({'task_id': tid, 'group': g,
                           'filled_fraction': float(df[gcols].replace('', np.nan).notna().any(axis=1).mean()),
                           'alone_auc': alone['auc'], 'alone_ci_low': alone['ci_low'], 'alone_ci_high': alone['ci_high'],
                           'drop_when_left_out': base - mean_auc(y, left), 'drop_when_shuffled': base - shuf})

    # Site or protocol (7b.10), binary tasks with known sites
    site = df.site.to_numpy()
    if k == 2 and pd.notna(site).any():
        has = pd.notna(site)
        ds, ys = df[has].reset_index(drop=True), y[has]
        sarr = ds.site.astype(str).to_numpy()
        srow = {'task_id': tid, 'n_patients_with_site': int(has.sum()), 'n_sites': len(np.unique(sarr))}
        # (a) site-only baseline
        est, _ = estimate(ds, ys, ['site'], 'lr')
        srow.update({'site_only_auc': est['auc'], 'site_only_ci_low': est['ci_low'], 'site_only_ci_high': est['ci_high']})
        # (b) site-held-out CV
        n_groups = len(np.unique(sarr))
        if n_groups >= 2:
            gkf = list(GroupKFold(min(N_SPLITS, n_groups)).split(np.zeros(len(ys)), ys, sarr))
            if all(len(np.unique(ys[tr])) == 2 for tr, _ in gkf):
                Pg, _ = cv_oof(ds, ys, SCANNER_COLS, 'lr', [gkf])
                lo_g, hi_g = boot_ci(ys, Pg)
                srow.update({'site_heldout_auc': mean_auc(ys, Pg), 'site_heldout_ci_low': lo_g, 'site_heldout_ci_high': hi_g})
            else:
                srow['site_heldout_note'] = 'a training fold had one class only'
        # (c) within-site AUC from the primary out-of-fold scores
        w = within_site_estimate(ys, P[:, has], sarr)
        srow.update({f'within_site_{k2}': v for k2, v in w.items()})
        srow['overall_auc_same_patients'] = mean_auc(ys, P[:, has])
        site_rows.append(srow)
        # (d) refit inside each site with >= 10 patients per class
        for st in np.unique(sarr):
            m = sarr == st
            if min((ys[m] == 0).sum(), (ys[m] == 1).sum()) >= MIN_PER_CLASS_SITE_REFIT:
                dd, yy = ds[m].reset_index(drop=True), ys[m]
                est, _ = estimate(dd, yy, SCANNER_COLS, 'lr')
                refit_rows.append({'task_id': tid, 'site': st, 'n_patients': int(m.sum()),
                                   'n_class_0': int((yy == 0).sum()), 'n_class_1': int((yy == 1).sum()), **est})

    primary_rows.append(row)
    oof_rows.append(oof)
    # save this task at once (resume point)
    oof.to_csv(PER_TASK / f'{stem}_oof.csv.gz', index=False)
    saved.write_text(json.dumps({'settings_key': SETTINGS_KEY, 'primary': row, 'groups': group_rows[n_group:],
                                 'sites': site_rows[n_site:], 'refit': refit_rows[n_refit:]}, default=_jsonable))
    print(f'{i:>2}/{len(family)} {tid:<44} AUC {base:.3f} [{lo:.3f}, {hi:.3f}]  p={p:.4f}  '
          f'({time.time() - t0:.0f} s)')
print(f'Audit finished in {(time.time() - t_all) / 60:.1f} min')

 1/26 lung histology (LUAD vs LUSC)                AUC 0.827 [0.745, 0.901]  p=0.0010  (147 s)
 2/26 renal subtype (KIRC vs KIRP)                 AUC 0.846 [0.759, 0.921]  p=0.0010  (170 s)
 3/26 CPTAC-CCRCC | grade                          AUC 0.431 [0.299, 0.578]  p=0.8050  (98 s)
 4/26 CPTAC-CCRCC | stage                          AUC 0.392 [0.264, 0.521]  p=0.8470  (96 s)
 5/26 CPTAC-PDA | grade                            AUC 0.369 [0.248, 0.501]  p=0.9790  (96 s)
 6/26 CPTAC-PDA | stage                            AUC 0.840 [0.734, 0.923]  p=0.0010  (95 s)
 7/26 CPTAC-UCEC | grade                           AUC 0.370 [0.251, 0.495]  p=0.9710  (106 s)
 8/26 Duke-Breast-Cancer-MRI | ER status           AUC 0.501 [0.464, 0.540]  p=0.4650  (178 s)
 9/26 Duke-Breast-Cancer-MRI | HER2 status         AUC 0.496 [0.452, 0.542]  p=0.3610  (183 s)
10/26 Duke-Breast-Cancer-MRI | PR status           AUC 0.512 [0.476, 0.547]  p=0.3840  (182 s)
11/26 Duke-Breast-Cancer-MRI | molecular subtype   AUC

## Multiple testing and flags (rule 7a.2)

In [10]:
# 10. Benjamini-Hochberg across the family, flags, and saving
def bh(p):
    p = np.asarray(p, float)
    order = np.argsort(p)
    q = np.empty_like(p)
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(p) / rank)
        q[j] = prev
    return q

primary = pd.DataFrame(primary_rows)
primary['perm_q_bh'] = bh(primary.perm_p)
primary['flagged'] = (primary.ci_low >= FLAG_LOWER_CI) & (primary.perm_q_bh < FLAG_Q)
primary = primary.sort_values('auc', ascending=False)
primary.to_csv(OUT / 'audit_primary.csv', index=False)
pd.DataFrame(group_rows).to_csv(OUT / 'audit_feature_groups.csv', index=False)
pd.DataFrame(site_rows).to_csv(OUT / 'audit_sites.csv', index=False)
pd.DataFrame(refit_rows, columns=['task_id', 'site', 'n_patients', 'n_class_0', 'n_class_1', 'auc', 'ci_low', 'ci_high']
             ).to_csv(OUT / 'audit_within_site_refit.csv', index=False)
pd.concat(oof_rows, ignore_index=True).to_csv(OUT / 'oof_predictions.csv.gz', index=False)

show(primary[['task_id', 'n_patients', 'auc', 'ci_low', 'ci_high', 'perm_p', 'perm_q_bh', 'flagged',
              'null_mean_auc', 'lgbm_auc', 'care_only_auc']], 40)

,task_id,n_patients,auc,ci_low,ci_high,perm_p,perm_q_bh,flagged,null_mean_auc,lgbm_auc,care_only_auc
1,renal subtype (KIRC vs KIRP),259,0.846500,0.758933,0.920989,0.001,0.006500,True,0.494625,0.848987,0.542944
5,CPTAC-PDA | stage,68,0.840122,0.734428,0.922889,0.001,0.006500,True,0.488033,0.785410,0.546910
0,lung histology (LUAD vs LUSC),104,0.827269,0.744907,0.900635,0.001,0.006500,True,0.493642,0.832836,0.653812
13,LIDC-IDRI | primary vs metastatic,94,0.790242,0.695271,0.874832,0.001,0.006500,True,0.486922,0.786867,0.721842
12,LIDC-IDRI | benign vs malignant,130,0.688298,0.594821,0.776647,0.005,0.026000,False,0.493865,0.720981,0.545154
22,TCGA-LIHC | grade,75,0.687500,0.569999,0.796000,0.008,0.034667,False,0.491093,0.607407,0.480556
18,TCGA-BLCA | stage,107,0.624612,0.519920,0.727755,0.052,0.150222,False,0.493034,0.645381,0.488248
21,TCGA-KIRC | stage,236,0.614588,0.543528,0.678828,0.010,0.037143,False,0.494953,0.601172,0.595969
16,NSCLC Radiogenomics | histology,199,0.604272,0.517244,0.687979,0.078,0.202800,False,0.492769,0.566064,0.585926
17,PROSTATEx | clinically significant,204,0.589104,0.521098,0.659616,0.046,0.149500,False,0.486788,0.605267,0.581407


In [11]:
# 11. Run record (with a checksum for every output file)
outputs = []
for p in sorted(q for q in OUT.rglob('*') if q.is_file()):
    if p.name == 'run_manifest.json':
        continue
    n_rows = len(pd.read_csv(p, low_memory=False)) if (p.suffix == '.csv' or p.name.endswith('.csv.gz')) else None
    outputs.append({'file': p.relative_to(OUT).as_posix(), 'bytes': p.stat().st_size, 'rows': n_rows, 'md5': md5(p)})
manifest = {
    'settings_key': SETTINGS_KEY,
    'notebook': '03_metadata_audit.ipynb', 'run_started_utc': RUN_STARTED,
    'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB, 'local_test_settings': TEST, 'input_notebook02_run': man02['run_started_utc'],
    'family': family,
    'settings': {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
                 'MIN_COUNT': MIN_COUNT, 'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'N_SHUFFLE': N_SHUFFLE,
                 'MIN_PER_CLASS_SITE_REFIT': MIN_PER_CLASS_SITE_REFIT, 'LR_PARAMS': LR_PARAMS,
                 'LGBM_PARAMS': LGBM_PARAMS, 'SCANNER_GROUPS': SCANNER_GROUPS, 'CARE_COLS': CARE_COLS},
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                 'scikit-learn': sklearn.__version__, 'lightgbm': lightgbm.__version__, 'joblib': joblib.__version__,
                 'cpus': os.cpu_count()},
    'outputs': outputs,
}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
show(pd.DataFrame(outputs)[lambda d: ~d.file.str.startswith('per_task/')])

,file,bytes,rows,md5
0,audit_feature_groups.csv,22939,156.0,52142bfef50bbf6c830513d3650e0924
1,audit_primary.csv,12227,26.0,ea83d4e57a688fc79c5e958df1040234
2,audit_sites.csv,2998,12.0,bcbed6dca85bbb03f4392639f899e9ba
3,audit_within_site_refit.csv,1615,15.0,5534e748bae5b1942faaf3663801bdd9
4,oof_predictions.csv.gz,683829,33195.0,1fac44600ce32b2647c35ac17f40b2d8


In [12]:
# 12. Plain-English summary, read back from the saved file
a = pd.read_csv(OUT / 'audit_primary.csv')
print(f"- Tasks audited: {len(a)}. Flagged (lower CI >= {FLAG_LOWER_CI} and q < {FLAG_Q}): {int(a.flagged.sum())}.")
for tt, g in a.groupby('task_type'):
    print(f"  - {tt}: {int(g.flagged.sum())} of {len(g)} flagged")
print(f"- Pipeline check: mean AUC under shuffled labels ranges {a.null_mean_auc.min():.3f} to "
      f"{a.null_mean_auc.max():.3f} (should be close to 0.5).")
print('\nNext: review the saved results and run log before writing up findings.')

- Tasks audited: 26. Flagged (lower CI >= 0.6 and q < 0.05): 4.
  - collection-defined (6b): 2 of 2 flagged
  - within collection (6a): 2 of 24 flagged
- Pipeline check: mean AUC under shuffled labels ranges 0.478 to 0.499 (should be close to 0.5).

Next: review the saved results and run log before writing up findings.
